In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

# Most reliable approach - resolves relative to the notebook file itself
# Walk up from cwd until we find the project root (identified by a known file)
project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
No data available. Run getDict() first.
{}

Out Players:
No data available. Run getDict() first.
{}
No data available. Run getDict() first.
No data available. Run getDict() first.


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
86,NaN,2025-26,1630573,Sam Hauser,Sam,1610612738,BOS,Boston Celtics,22501168,2026-04-09T00:00:00,BOS @ NYK,L,31.483333,2,7,0.286,2,6,0.333,0,0,0.000,0,2,2,3,0,0,0,0,0,0,6,1,12.9,0,0,13.0,1,31:29,1,118.6,119.0,119.0,114.9,115.3,115.3,3.8,3.7,3.7,0.143,0.00,30.0,0.000,0.071,0.032,0.0,0.0,0.429,0.429,0.101,0.104,89.46,89.19,74.33,89.19,0.047,58,2.0,7.0,38,84,0.452,16,43,0.372,14,16,0.875,13,29,42,23,11.0,4,1,2,16,17,106,-6.0,119.0,120.5,126.4,127.3,-7.4,-6.8,0.605,2.09,18.0,0.354,0.789,0.547,0.125,0.548,0.582,88.8,88.0,73.33,88,0.453,1610612752,NYK,New York Knicks,43,80,0.538,15,35,0.429,11,15,0.733,5,25,30,29,7.0,9,2,1,17,16,112,6.0,126.4,127.3,119.0,120.5,7.4,6.8,0.674,4.14,23.2,0.211,0.646,0.453,0.080,0.631,0.647,88.8,88.0,73.33,88,0.547,F,PF,28.0
87,NaN,2025-26,1629020,Jarred Vanderbilt,Jarred,1610612747,LAL,Los Angeles Lakers,22501170,2026-04-09T00:00:00,LAL @ GSW,W,25.596667,1,3,0.333,0,2,0.000,0,0,0.000,1,5,6,5,4,0,0,0,0,0,2,15,12.7,0,0,13.0,1,25:36,1,122.5,129.4,129.4,101.6,98.1,98.1,20.9,31.3,31.3,0.192,1.25,41.7,0.050,0.238,0.146,33.3,33.3,0.333,0.333,0.121,0.121,97.59,96.58,80.48,96.58,0.051,51,1.0,3.0,49,80,0.613,16,29,0.552,5,8,0.625,8,25,33,37,19.0,14,3,2,13,6,119,16.0,125.9,129.3,114.1,112.0,11.8,17.4,0.755,1.95,26.4,0.324,0.595,0.474,0.207,0.713,0.712,92.4,92.0,76.67,92,0.570,1610612744,GSW,Golden State Warriors,41,81,0.506,9,30,0.300,12,12,1.000,15,23,38,24,19.0,8,2,3,6,13,103,-16.0,114.1,112.0,125.9,129.3,-11.8,-17.4,0.585,1.26,18.0,0.405,0.676,0.526,0.207,0.562,0.597,92.4,92.0,76.67,92,0.430,NaN,PF,26.0
88,NaN,2025-26,1642880,Kam Jones,Kam,1610612754,IND,Indiana Pacers,22501167,2026-04-09T00:00:00,IND @ BKN,W,21.733333,2,7,0.286,0,2,0.000,0,0,0.000,1,2,3,6,4,0,0,1,2,1,4,9,12.6,0,0,13.0,1,21:44,1,116.8,119.6,119.6,99.7,97.9,97.9,17.2,21.7,21.7,0.286,1.50,35.3,0.045,0.083,0.065,23.5,23.5,0.286,0.286,0.208,0.203,102.96,102.70,85.58,102.70,0.016,46,2.0,7.0,51,98,0.520,8,31,0.258,13,18,0.722,13,53,66,43,12.0,4,5,5,16,21,123,29.0,117.2,118.3,87.6,90.4,29.7,27.9,0.843,3.58,26.9,0.260,0.828,0.579,0.115,0.561,0.581,106.1,104.0,86.67,104,0.714,1610612751,BKN,Brooklyn Nets,37,96,0.385,8,38,0.211,12,19,0.632,7,36,43,20,10.0,2,5,5,21,16,94,-29.0,87.6,90.4,117.2,118.3,-29.7,-27.9,0.541,2.00,14.8,0.172,0.740,0.421,0.096,0.427,0.450,106.1,104.0,86.67,104,0.286,NaN,SG,23.0
60,NaN,2025-26,1630551,Justin Champagnie,Justin,1610612764,WAS,Washing

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260409_224406.json


,home_team,away_team,commence_time,bookmakers
0,Atlanta Hawks,Cleveland Cavaliers,2026-04-10 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Charlotte Hornets,Detroit Pistons,2026-04-10 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Washington Wizards,Miami Heat,2026-04-10 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Boston Celtics,New Orleans Pelicans,2026-04-10 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Indiana Pacers,Philadelphia 76ers,2026-04-10 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-09 22:42:00
US latest pull: 2026-04-09 22:44:06


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,James Harden,Over,21.5,-137,2026-04-10,2026-04-10T05:41:41Z,2026-04-09 22:42:00
1,Underdog,player_points,James Harden,Under,21.5,-137,2026-04-10,2026-04-10T05:41:41Z,2026-04-09 22:42:00
2,Underdog,player_points,Evan Mobley,Over,17.5,-137,2026-04-10,2026-04-10T05:41:41Z,2026-04-09 22:42:00
3,Underdog,player_points,Evan Mobley,Under,17.5,-137,2026-04-10,2026-04-10T05:41:41Z,2026-04-09 22:42:00
4,Underdog,player_points,Jonathan Kuminga,Over,12.5,-137,2026-04-10,2026-04-10T05:41:41Z,2026-04-09 22:42:00


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] Derrick Jones: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,James Harden,AST,5.71,30.03,39.90,0.1553,0.1960,0.3426,0.89,5.89,13.67,"[0.2608922515001304, 0.1716443529007895, 0.217..."
1,Evan Mobley,AST,9.23,25.29,36.99,0.0502,0.1250,0.2087,0.46,3.16,7.72,"[0.0321750321750321, 0.0355618776671408, 0.126..."
2,Jalen Johnson,AST,10.58,31.40,39.53,0.0924,0.1584,0.2739,0.98,4.97,10.83,"[0.2614758861127251, 0.1810188776829583, 0.211..."
3,CJ McCollum,AST,14.95,28.30,36.75,0.0704,0.1538,0.2586,1.05,4.35,9.51,"[0.1428571428571428, 0.0662690523525513, 0.074..."
4,Max Strus,AST,14.46,23.99,30.66,0.0583,0.1027,0.2719,0.84,2.46,8.34,"[0.2534854245880861, 0.0857632933104631, 0.093..."
5,Onyeka Okongwu,AST,14.47,28.36,36.89,0.0329,0.0848,0.1672,0.48,2.41,6.17,"[0.1728907330567081, 0.0614817091915155, 0.0, ..."
6,Miles Bridges,AST,14.23,29.46,37.44,0.0488,0.1094,0.2058,0.69,3.22,7.71,"[0.0899550224887556, 0.159846547314578, 0.0591..."
7,Ausar Thompson,AST,17.84,26.61,35.30,0.0727,0.1249,0.2112,1.30,3.32,7.46,"[0.1921229586935638, 0.0, 0.124275062137531, 0..."
8,Daniss Jenkins,AST,20.29,27.04,33.32,0.0877,0.1865,0.2919,1.78,5.04,9.73,"[0.2706359945872801, 0.1529051987767584, 0.328..."
9,Derrick White,AST,11.98,29.23,38.65,0.0694,0.1435,0.2516,0.83,4.19,9.72,"[0.1287415513356936, 0.1404099971918, 0.175087..."


### Get Line Probabilities

In [8]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
4,Max Strus,AST,2.5,14.46,23.99,30.66,0.84,2.46,8.34,0.470,0.530
19,Immanuel Quickley,AST,4.5,14.32,25.52,35.19,1.03,4.39,9.98,0.508,0.492
68,James Harden,PTS,21.5,5.71,30.03,39.90,2.40,17.62,38.82,0.311,0.689
63,Sion James,REB,3.5,16.61,22.89,29.11,0.58,3.07,8.08,0.535,0.465
135,Stephon Castle,PTS,15.5,15.30,28.95,35.15,6.08,18.47,33.17,0.605,0.395
107,Bez Mbeng,PTS,14.5,31.41,34.72,39.63,6.28,7.08,18.37,0.167,0.833
70,Jonathan Kuminga,PTS,12.5,16.56,21.88,29.71,5.79,13.40,27.66,0.498,0.502
112,Scoot Henderson,PTS,14.5,19.97,29.42,37.12,6.53,15.06,33.39,0.702,0.298
110,Darius Garland,PTS,20.5,15.53,26.42,35.50,6.45,16.72,36.58,0.414,0.586
127,Jakob Poeltl,PTS,9.5,14.31,24.73,32.80,4.88,11.37,26.10,0.804,0.196


In [9]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
100,Luguentz Dort,PTS,8.5,12.15,25.80,34.87,2.23,9.82,22.31,0.569,0.431,PTS,Underdog,Denver Nuggets,10.5,231.5,116.2,21.0,99.42,20.0,-137.0,-137.0,0.578,0.578,7.9,7.0,4.98,-0.6,-1.5,0.120,0.452,0.548,-21.81,-5.20,0.4,0.4,0.33,0.49,21.46,2.49,0.14,0.05,9.71,7.0
27,Nickeil Alexander-Walker,REB,3.5,15.84,29.42,39.26,0.40,3.24,9.27,0.443,0.557,REB,Underdog,Cleveland Cavaliers,-7.5,233.5,114.0,15.0,100.68,13.0,-103.0,-125.0,0.507,0.556,3.5,4.0,1.08,0.0,0.5,0.000,0.500,0.500,-1.46,-10.00,0.8,0.7,0.60,0.38,34.89,4.96,0.22,0.03,3.80,5.0
87,Saddiq Bey,PTS,18.5,16.51,29.19,36.06,4.14,15.87,36.43,0.453,0.547,PTS,Underdog,Boston Celtics,16.5,223.5,111.8,4.0,95.35,30.0,-137.0,-137.0,0.578,0.578,20.3,19.0,4.85,1.8,0.5,-0.371,0.645,0.355,11.58,-38.59,0.6,0.6,0.60,0.44,32.94,3.58,0.23,0.04,11.00,1.0
13,Cameron Johnson,AST,2.5,16.47,28.02,36.55,0.56,2.36,6.72,0.515,0.485,AST,Underdog,Oklahoma City Thunder,-10.5,231.5,105.9,1.0,100.37,15.0,-137.0,-137.0,0.578,0.578,2.9,2.5,2.13,0.4,0.0,-0.188,0.575,0.425,-0.53,-26.48,0.6,0.5,0.53,0.56,30.71,5.78,0.15,0.02,2.75,4.0
74,LaMelo Ball,PTS,21.5,15.42,29.19,35.78,7.60,20.84,38.22,0.506,0.494,PTS,Underdog,Detroit Pistons,-3.5,223.5,108.9,2.0,99.87,19.0,-119.0,103.0,0.543,0.493,22.8,20.0,7.84,1.3,-1.5,-0.166,0.566,0.434,4.16,-11.90,0.4,0.4,0.40,0.49,30.19,4.90,0.30,0.06,22.00,4.0


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
48,Franz Wagner,REB,3.5,14.36,27.44,35.44,0.71,4.10,10.77,0.573,0.427,REB,PrizePicks,Chicago Bulls,-14.5,242.5,117.2,22.0,103.06,3.0,-132.0,100.0,0.569,0.500,3.3,3.5,2.75,-0.2,0.0,0.073,0.471,0.529,-17.22,5.80,0.4,0.5,0.67,0.83,21.01,7.09,0.27,0.05,4.20,5.0
6,Miles Bridges,AST,2.5,14.23,29.46,37.44,0.69,3.22,7.71,0.514,0.486,AST,PrizePicks,Detroit Pistons,-3.5,223.5,108.9,2.0,99.87,19.0,-105.0,100.0,0.512,0.500,2.7,2.0,1.89,0.2,-0.5,-0.106,0.542,0.458,5.82,-8.40,0.4,0.4,0.40,0.61,28.30,3.44,0.21,0.06,2.60,5.0
120,Max Strus,PTS,10.5,14.46,23.99,30.66,3.03,9.04,22.43,0.469,0.530,PTS,PrizePicks,Atlanta Hawks,7.5,233.5,112.7,9.0,102.48,6.0,-120.0,-106.0,0.545,0.515,12.2,9.5,9.93,1.7,-1.0,-0.171,0.568,0.432,4.13,-16.05,0.6,0.5,0.47,0.42,24.70,2.93,0.17,0.05,9.50,2.0
53,Donovan Clingan,REB,12.5,17.81,25.79,34.48,4.29,11.42,20.21,0.429,0.571,REB,PrizePicks,Los Angeles Clippers,-2.0,224.5,115.1,18.0,97.35,28.0,-137.0,-137.0,0.578,0.578,11.3,12.0,4.24,-0.7,0.0,0.165,0.434,0.566,-24.92,-2.09,0.0,0.4,0.47,0.28,26.74,3.80,0.16,0.05,7.00,4.0
95,Max Christie,PTS,10.5,13.43,27.55,34.35,3.03,10.79,20.65,0.417,0.583,PTS,PrizePicks,San Antonio Spurs,17.0,236.5,110.0,3.0,100.73,12.0,-137.0,-137.0,0.578,0.578,9.4,8.5,5.19,-1.1,-2.0,0.212,0.416,0.584,-28.04,1.03,0.6,0.4,0.53,0.53,28.37,3.40,0.15,0.05,12.12,8.0


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
111,Jrue Holiday,PTS,17.5,16.26,29.15,38.46,4.92,13.80,32.02,0.405,0.595,PTS,Betr DFS,Los Angeles Clippers,-2.0,224.5,115.1,18.0,97.35,28.0,-112.0,-112.0,0.528,0.528,16.8,13.0,7.47,-0.7,-4.5,0.094,0.463,0.537,-12.36,1.65,0.6,0.4,0.33,0.27,31.05,5.28,0.22,0.06,18.67,3.0
136,De'Aaron Fox,PTS,14.5,15.03,28.81,37.04,8.01,20.89,37.06,0.611,0.389,PTS,Betr DFS,Dallas Mavericks,-17.0,236.5,115.2,19.0,102.56,4.0,-137.0,-137.0,0.578,0.578,15.5,14.0,5.84,1.0,-0.5,-0.171,0.568,0.432,-1.74,-25.27,0.4,0.3,0.53,0.73,28.69,5.89,0.24,0.06,22.20,5.0
68,James Harden,PTS,21.5,5.71,30.03,39.90,2.40,17.62,38.82,0.311,0.689,PTS,Betr DFS,Atlanta Hawks,7.5,233.5,112.7,9.0,102.48,6.0,-116.0,-110.0,0.537,0.524,21.5,19.5,6.72,0.0,-2.0,0.000,0.500,0.500,-6.90,-4.55,0.2,0.3,0.33,0.52,35.31,4.29,0.24,0.07,23.60,5.0
59,Jalen Johnson,REB,10.5,10.58,31.40,39.53,1.59,9.51,17.78,0.331,0.669,REB,Betr DFS,Cleveland Cavaliers,-7.5,233.5,114.0,15.0,100.68,13.0,-135.0,105.0,0.574,0.488,9.4,11.0,3.17,-0.1,1.5,0.032,0.487,0.513,-15.23,5.16,1.0,0.6,0.60,0.59,35.61,4.17,0.26,0.04,10.80,5.0
60,Dyson Daniels,REB,7.5,14.95,29.88,38.18,1.08,5.27,12.35,0.342,0.658,REB,Betr DFS,Cleveland Cavaliers,-7.5,233.5,114.0,15.0,100.68,13.0,115.0,-150.0,0.465,0.600,7.5,6.5,3.78,0.0,-1.0,0.000,0.500,0.500,7.50,-16.67,0.4,0.3,0.33,0.34,32.59,5.81,0.16,0.03,7.17,6.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
25,LaMelo Ball,AST,7.5,15.42,29.19,35.78,2.12,5.95,12.60,0.404,0.596,AST,DraftKings Pick6,Detroit Pistons,-3.5,223.5,108.9,2.0,99.87,19.0,100.0,-128.0,0.500,0.561,7.2,8.0,2.44,-0.3,0.5,0.123,0.451,0.549,-9.80,-2.21,0.8,0.6,0.47,0.45,30.19,4.90,0.30,0.06,6.75,4.0
84,Kon Knueppel,PTS,17.5,22.32,31.33,37.43,8.93,19.49,33.93,0.451,0.549,PTS,DraftKings Pick6,Detroit Pistons,-3.5,223.5,108.9,2.0,99.87,19.0,-113.0,-108.0,0.531,0.519,14.5,13.0,5.72,-3.0,-4.5,0.524,0.300,0.700,-43.45,34.81,0.4,0.3,0.40,0.58,30.12,5.95,0.22,0.02,19.50,2.0
112,Scoot Henderson,PTS,14.5,19.97,29.42,37.12,6.53,15.06,33.39,0.702,0.298,PTS,DraftKings Pick6,Los Angeles Clippers,-2.0,224.5,115.1,18.0,97.35,28.0,-122.0,-102.0,0.550,0.505,15.7,14.5,4.60,1.2,0.0,-0.261,0.603,0.397,9.73,-21.38,0.8,0.5,0.53,0.38,26.93,5.75,0.25,0.03,15.00,3.0
69,Evan Mobley,PTS,18.5,9.23,25.29,36.99,3.69,13.85,32.30,0.385,0.615,PTS,DraftKings Pick6,Atlanta Hawks,7.5,233.5,112.7,9.0,102.48,6.0,100.0,-127.0,0.500,0.559,19.2,20.5,8.59,0.7,2.0,-0.081,0.532,0.468,6.40,-16.35,0.6,0.6,0.53,0.47,30.40,4.41,0.22,0.06,19.67,6.0
0,James Harden,AST,6.5,5.71,30.03,39.90,0.89,5.89,13.67,0.429,0.571,AST,DraftKings Pick6,Atlanta Hawks,7.5,233.5,112.7,9.0,102.48,6.0,-125.0,-105.0,0.556,0.512,8.0,7.0,3.80,1.5,0.5,-0.395,0.654,0.346,17.72,-32.45,0.4,0.7,0.67,0.74,35.31,4.29,0.24,0.07,9.20,5.0


In [13]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
36,Devin Vassell,REB,3.5,15.47,28.60,36.44,0.44,3.93,10.39,0.570,0.430,REB,Underdog,Dallas Mavericks,-17.0,236.5,115.2,19.0,102.56,4.0,-137.0,-137.0,0.578,0.578,5.1,4.5,2.81,1.6,1.0,-0.569,0.715,0.285,23.69,-50.70,0.8,0.7,0.60,0.52,29.20,4.95,0.16,0.04,5.60,5.0
81,Jalen Duren,PTS,18.5,13.87,28.50,37.46,4.48,13.44,32.06,0.469,0.531,PTS,PrizePicks,Charlotte Hornets,3.5,223.5,113.5,11.0,97.68,26.0,-137.0,-137.0,0.578,0.578,21.7,21.5,6.34,2.7,2.5,-0.426,0.665,0.335,15.04,-42.05,0.6,0.7,0.73,0.25,31.69,5.42,0.23,0.06,10.83,6.0
82,Duncan Robinson,PTS,9.5,15.46,24.49,34.92,3.85,10.17,23.81,0.758,0.242,PTS,PrizePicks,Charlotte Hornets,3.5,223.5,113.5,11.0,97.68,26.0,-118.0,-110.0,0.541,0.524,13.7,13.0,4.00,4.2,3.5,-1.050,0.853,0.147,57.59,-71.94,1.0,0.9,0.80,0.58,27.16,2.65,0.16,0.04,12.67,6.0
81,Jalen Duren,PTS,18.5,13.87,28.50,37.46,4.48,13.44,32.06,0.469,0.531,PTS,Underdog,Charlotte Hornets,3.5,223.5,113.5,11.0,97.68,26.0,-137.0,-137.0,0.578,0.578,21.7,21.5,6.34,3.2,3.0,-0.505,0.693,0.307,19.88,-46.89,0.6,0.7,0.73,0.28,31.69,5.42,0.23,0.06,10.83,6.0
111,Jrue Holiday,PTS,17.5,16.26,29.15,38.46,4.92,13.80,32.02,0.405,0.595,PTS,Betr DFS,Los Angeles Clippers,-2.0,224.5,115.1,18.0,97.35,28.0,-112.0,-112.0,0.528,0.528,16.8,13.0,7.47,-0.7,-4.5,0.094,0.463,0.537,-12.36,1.65,0.6,0.4,0.33,0.27,31.05,5.28,0.22,0.06,18.67,3.0


### Get top EVs for 2 legs

In [14]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 91  |  Pairs: 214  |  Slate: 8  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [15]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 57  |  Pairs: 104  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [16]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 21  |  Pairs: 3  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 55  |  Pairs: 62  |  Slate: 3  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [18]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 91  |  Triples: 4407  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [19]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 57  |  Triples: 1235  |  Slate: 4  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 55  |  Triples: 514  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 21  |  Triples: 3  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
